In [1]:
import ast
import json
import numpy as np
import os
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import fasttext
import fasttext.util
from sklearn.model_selection import GridSearchCV
from itertools import product

DOWNLOAD AND REDUCE FASTTEXT

In [ ]:
# Download the pre-trained FastText model for English
fasttext.util.download_model("en", if_exists="ignore")

# Load the pre-trained model
ft = fasttext.load_model("cc.en.300.bin")

# Reduce the dimensions of the model
ft_reduced = fasttext.util.reduce_model(ft, 100)

# Save the reduced model
ft_reduced.save_model("cc.en.300_reduced.bin")

LOAD THE FILE AND CREATE A DATAFRAME

In [ ]:
with open("../../data/tacred/json/train.json", "r") as file:
    data_train = json.load(file)

with open("../../data/tacred/json/test.json", "r") as file:
    data_test = json.load(file)

with open("../../data/tacred/json/dev.json", "r") as file:
    data_dev = json.load(file)

In [2]:
# Read data from the "dev.json" file
with open("train.json", "r") as file:
    data_train = json.load(file)
# Read data from the "dev.json" file
with open("test.json", "r") as file:
    data_test = json.load(file)
# Read data from the "dev.json" file
with open("dev.json", "r") as file:
    data_dev = json.load(file)

In [4]:
# Create a DataFrame from the loaded data
df_test = pd.DataFrame(data_test)
df_train = pd.DataFrame(data_train)
df_dev = pd.DataFrame(data_dev)

LOAD FASTTEXT PRETRAINED MODEL

In [5]:
# Load the reduced model
ft = fasttext.load_model("cc.en.300_reduced.bin")

EMBEDDINGS CREATION

Creation of embeddingd for each token using FastText pretrained model reduced to 100 dimensions 

In [6]:
# Generate embeddings for each token in the list
def get_embeddings(tokens):
    embeddings = []
    max_length = max(len(token_list) for token_list in tokens)

    for token_list in tokens:
        token_embeddings = [ft.get_word_vector(token) for token in token_list]
        # Padding or truncating embeddings to match the maximum length
        padding = [np.zeros_like(token_embeddings[0])] * (max_length - len(token_list))
        padded_embeddings = (
            token_embeddings + padding
            if len(token_list) < max_length
            else token_embeddings[:max_length]
        )
        embeddings.append(padded_embeddings)

    return embeddings

In [7]:
df_train["embeddings"] = get_embeddings(df_train["token"])
df_train["mean_embeddings"] = df_train["embeddings"].apply(np.mean)

In [8]:
df_test["embeddings"] = get_embeddings(df_test["token"])
df_test["mean_embeddings"] = df_test["embeddings"].apply(np.mean)

In [9]:
df_dev["embeddings"] = get_embeddings(df_dev["token"])
df_dev["mean_embeddings"] = df_dev["embeddings"].apply(np.mean)

COMPUTE THE DISTANCE BETWEEN SUBJ AND OBJ

As additional feature, it computes the distance between subject and object

In [10]:
# Calculate the distance between subject and object
def calculate_distance(df):
    df["subj_midpoint"] = (df["subj_start"] + df["subj_end"]) / 2
    df["obj_midpoint"] = (df["obj_start"] + df["obj_end"]) / 2
    df["subj_obj_distance"] = df["obj_midpoint"] - df["subj_midpoint"]
    df = df.drop(columns=["subj_midpoint", "obj_midpoint"])
    return df

In [11]:
df_train = calculate_distance(df_train)

In [12]:
df_test = calculate_distance(df_test)

In [13]:
df_dev = calculate_distance(df_dev)

COMPUTE COSINE SIMILARITY BETWEEN SUBJ AND OBJ

Creation of a column for subject and one for object in order to compute the cosine similarity.

In [14]:
def extract_subject_object(row):
    subj_start = row["subj_start"]
    subj_end = row["subj_end"]
    obj_start = row["obj_start"]
    obj_end = row["obj_end"]

    subject = " ".join(row["token"][subj_start : subj_end + 1])
    object_ = " ".join(row["token"][obj_start : obj_end + 1])

    return pd.Series({"Subject": subject, "Object": object_})

In [15]:
df_train[["Subject", "Object"]] = df_train.apply(extract_subject_object, axis=1)

In [16]:
df_test[["Subject", "Object"]] = df_test.apply(extract_subject_object, axis=1)

In [17]:
df_dev[["Subject", "Object"]] = df_dev.apply(extract_subject_object, axis=1)

In [18]:
# Calculate cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_train["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_train["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_train["word_similarity"] = similarities

In [19]:
# Calculate cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_test["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_test["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_test["word_similarity"] = similarities

In [20]:
# Calculate cosine similarity between corresponding pairs of subject and object embeddings
subject_embeddings = [ft.get_word_vector(word) for word in df_dev["Subject"]]
object_embeddings = [ft.get_word_vector(word) for word in df_dev["Object"]]

similarities = [
    cosine_similarity([sub_emb], [obj_emb])[0][0]
    for sub_emb, obj_emb in zip(subject_embeddings, object_embeddings)
]

df_dev["word_similarity"] = similarities

ONE HOT ENCODING OF OBJ_TYPES AND SUBJ_TYPES

Creation of a column for each subject and object type and performs one-hot encoding on them, creating binary columns to represent unique categories within each specified column

In [21]:
# Add additional binary columns for each unique category in the specified columns
def perform_one_hot_encoding(df, columns=["subj_type", "obj_type"]):
    for column in columns:
        dummies = pd.get_dummies(df[column], prefix=column)
        df = pd.concat([df, dummies], axis=1)
    return df

In [22]:
df_train = perform_one_hot_encoding(df_train, columns=["subj_type", "obj_type"])

In [23]:
df_test = perform_one_hot_encoding(df_test, columns=["subj_type", "obj_type"])

In [24]:
df_dev = perform_one_hot_encoding(df_dev, columns=["subj_type", "obj_type"])

ONE HOT ENCODING OF STANDFORD POS

This function conducts one-hot encoding on stanford_POS column that contains lists of tags. It creates binary columns for each unique tag value, assigning 1 to cells where a tag is present and 0 otherwise. The resulting DataFrame incorporates the new one-hot encoded columns for each unique tag value.

In [25]:
def one_hot_encoding_pos(df, column_name):
    unique_tags = set(tag for tag_list in df[column_name] for tag in tag_list)

    for tag_value in unique_tags:
        df[tag_value] = 0

    for index, row in df.iterrows():
        for tag in row[column_name]:
            df.at[index, tag] = 1

    return df

In [26]:
df_train = one_hot_encoding_pos(df_train, "stanford_pos")

In [27]:
df_test = one_hot_encoding_pos(df_test, "stanford_pos")

In [28]:
df_dev = one_hot_encoding_pos(df_dev, "stanford_pos")

DELETE THE USELESS COLUMNS FOR THE MODEL

In [29]:
df_train = df_train.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

In [30]:
df_test = df_test.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

In [31]:
df_dev = df_dev.drop(
    columns=[
        "id",
        "docid",
        "embeddings",
        "token",
        "subj_start",
        "subj_end",
        "obj_start",
        "obj_end",
        "subj_type",
        "obj_type",
        "stanford_ner",
        "stanford_pos",
        "stanford_head",
        "stanford_deprel",
        "Subject",
        "Object",
    ]
)

SCORE FUNCTION

In [32]:
import argparse
import sys
from collections import Counter

NO_RELATION = "no_relation"


def score(key, prediction, verbose=False):
    correct_by_relation = Counter()
    guessed_by_relation = Counter()
    gold_by_relation = Counter()

    # Loop over the data to compute a score
    for row in range(len(key)):
        gold = key[row]
        guess = prediction[row]

        if gold == NO_RELATION and guess == NO_RELATION:
            pass
        elif gold == NO_RELATION and guess != NO_RELATION:
            guessed_by_relation[guess] += 1
        elif gold != NO_RELATION and guess == NO_RELATION:
            gold_by_relation[gold] += 1
        elif gold != NO_RELATION and guess != NO_RELATION:
            guessed_by_relation[guess] += 1
            gold_by_relation[gold] += 1
            if gold == guess:
                correct_by_relation[guess] += 1

    # Print verbose information
    if verbose:
        print("Per-relation statistics:")
        relations = gold_by_relation.keys()
        longest_relation = 0
        for relation in sorted(relations):
            longest_relation = max(len(relation), longest_relation)
        for relation in sorted(relations):
            # (compute the score)
            correct = correct_by_relation[relation]
            guessed = guessed_by_relation[relation]
            gold = gold_by_relation[relation]
            prec = 1.0
            if guessed > 0:
                prec = float(correct) / float(guessed)
            recall = 0.0
            if gold > 0:
                recall = float(correct) / float(gold)
            f1 = 0.0
            if prec + recall > 0:
                f1 = 2.0 * prec * recall / (prec + recall)
            # (print the score)
            sys.stdout.write(("{:<" + str(longest_relation) + "}").format(relation))
            sys.stdout.write("  P: ")
            if prec < 0.1:
                sys.stdout.write(" ")
            if prec < 1.0:
                sys.stdout.write(" ")
            sys.stdout.write("{:.2%}".format(prec))
            sys.stdout.write("  R: ")
            if recall < 0.1:
                sys.stdout.write(" ")
            if recall < 1.0:
                sys.stdout.write(" ")
            sys.stdout.write("{:.2%}".format(recall))
            sys.stdout.write("  F1: ")
            if f1 < 0.1:
                sys.stdout.write(" ")
            if f1 < 1.0:
                sys.stdout.write(" ")
            sys.stdout.write("{:.2%}".format(f1))
            sys.stdout.write("  #: %d" % gold)
            sys.stdout.write("\n")
        print("")

    # Print the aggregate score
    if verbose:
        print("Final Score:")
    prec_micro = 1.0
    if sum(guessed_by_relation.values()) > 0:
        prec_micro = float(sum(correct_by_relation.values())) / float(
            sum(guessed_by_relation.values())
        )
    recall_micro = 0.0
    if sum(gold_by_relation.values()) > 0:
        recall_micro = float(sum(correct_by_relation.values())) / float(
            sum(gold_by_relation.values())
        )
    f1_micro = 0.0
    if prec_micro + recall_micro > 0.0:
        f1_micro = 2.0 * prec_micro * recall_micro / (prec_micro + recall_micro)
    print("Precision (micro): {:.3%}".format(prec_micro))
    print("   Recall (micro): {:.3%}".format(recall_micro))
    print("       F1 (micro): {:.3%}".format(f1_micro))
    return prec_micro, recall_micro, f1_micro

RANDOM FOREST CALSSIFIER

Prepare data to be given as input to the Random Forest Classieier and then try different classifiers with different combinations of values for number of estimators and for max depth. It saves the parameters with which it achieves the best result and then train a new classifier with that parameter.The dataset is balanced thanks to class_weight parameter of RandomForestClassifier.

In [33]:
X_train = df_train.drop(columns=["relation"])
y_train = df_train["relation"]

X_test = df_test.drop(columns=["relation"])
y_test = df_test["relation"]

X_dev = df_dev.drop(columns=["relation"])
y_dev = df_dev["relation"]

In [34]:
n_estimators_values = [100, 200, 300]
max_depth_values = [None, 10, 20]
best_f1_micro = 0
dev = y_dev.tolist()

# Iterate over all combinations of hyperparameter values
for n_estimators, max_depth in product(n_estimators_values, max_depth_values):
    rf_classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight="balanced",
        random_state=42,
    )

    rf_classifier.fit(X_train, y_train)
    predictions = rf_classifier.predict(X_dev)
    predictions = predictions.tolist()

    prec_micro, recall_micro, current_f1_micro = score(
        y_dev, predictions, verbose=False
    )

    if current_f1_micro > best_f1_micro:
        best_f1_micro = current_f1_micro
        best_params = {
            "n_estimators_values": n_estimators,
            "max_depth_values": max_depth,
        }

# Create the Random Forest classifier with the best parameters
best_rf_classifier = RandomForestClassifier(
    n_estimators=best_params["n_estimators_values"],
    max_depth=best_params["max_depth_values"],
    class_weight="balanced",
    random_state=42,
)

best_rf_classifier.fit(X_train, y_train)

best_predictions = best_rf_classifier.predict(X_test)

best_predictions = best_predictions.tolist()
y_test = y_test.tolist()

score(y_test, best_predictions, verbose=False)

C:\Users\Hp\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 73.732%
   Recall (micro): 22.461%
       F1 (micro): 34.433%


C:\Users\Hp\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 18.823%
   Recall (micro): 76.104%
       F1 (micro): 30.182%


C:\Users\Hp\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


...

C:\Users\Hp\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 18.609%
   Recall (micro): 76.104%
       F1 (micro): 29.906%


C:\Users\Hp\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names must be in the same order as they were in fit.

  warnings.warn(message, FutureWarning)


Precision (micro): 39.381%
   Recall (micro): 57.138%
       F1 (micro): 46.626%
Precision (micro): 38.279%
   Recall (micro): 59.278%
       F1 (micro): 46.519%


(0.382792775296174, 0.5927819548872181, 0.4651876327590276)

ERROS ANALYSIS



In [35]:
from collections import Counter

associated_values_founded_by = []
associated_values_member_of = []

for index, label in enumerate(y_test):
    if label == "org:founded_by":
        associated_values_founded_by.append(best_predictions[index])
    elif label == "org:member_of":
        associated_values_member_of.append(best_predictions[index])
    else:
        pass  # Do nothing when the value is neither 'org:founded_by' nor 'org:member_of'

# Count for 'org:founded_by'
count_associated_values_founded_by = sum(
    1 for value in associated_values_founded_by if value is not None
)
print(
    "Number of associated values for 'org:founded_by':",
    count_associated_values_founded_by,
)

# Remove None values from the list to get only the associated values for 'org:founded_by'
associated_values_founded_by_without_none = [
    value for value in associated_values_founded_by if value is not None
]

# Count the distribution of associated values for 'org:founded_by'
distribution_founded_by = Counter(associated_values_founded_by_without_none)

# Print the distribution of associated values for 'org:founded_by'
print("Distribution of associated values for 'org:founded_by':")
for value, count in distribution_founded_by.items():
    print(f"Value: {value}, Frequency: {count}")

# Count for 'org:member_of'
count_associated_values_member_of = sum(
    1 for value in associated_values_member_of if value is not None
)
print(
    "\nNumber of associated values for 'org:member_of':",
    count_associated_values_member_of,
)

# Remove None values from the list to get only the associated values for 'org:member_of'
associated_values_member_of_without_none = [
    value for value in associated_values_member_of if value is not None
]

# Count the distribution of associated values for 'org:member_of'
distribution_member_of = Counter(associated_values_member_of_without_none)

# Print the distribution of associated values for 'org:member_of'
print("Distribution of associated values for 'org:member_of':")
for value, count in distribution_member_of.items():
    print(f"Value: {value}, Frequency: {count}")

Number of associated values for 'org:founded_by': 68
Distribution of associated values for 'org:founded_by':
Value: org:top_members/employees, Frequency: 52
Value: no_relation, Frequency: 16

Number of associated values for 'org:member_of': 18
Distribution of associated values for 'org:member_of':
Value: no_relation, Frequency: 14
Value: org:country_of_headquarters, Frequency: 3
Value: org:parents, Frequency: 1
